In [1]:
import torch
import torch.nn as nn

class CrossAttnPredictor(nn.Module):
    def __init__(self, embed_dim=192, num_heads=6, depth=4, num_shared_queries=8):
        """
        embed_dim: 192 (kyunki hamara tiny ViT 192-dimension features output kar raha hai)
        num_heads & depth: Transformer ke parameters (X-JEPA uses a lightweight predictor)
        num_shared_queries: Semantic anchors jo Optical aur SAR ke beech translation karte hain
        """
        super().__init__()
        self.embed_dim = embed_dim
        
        # 1. Learnable MASK Token (Yeh hidden SAR patches ko represent karta hai)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # 2. Learnable Shared Queries (X-JEPA paper ka main innovation)
        self.shared_queries = nn.Parameter(torch.zeros(1, num_shared_queries, embed_dim))
        
        # 3. Transformer Decoder (Cross-Attention)
        # Yeh MASK tokens ko Optical Context ke saath compare karke unki value predict karega
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=num_heads, batch_first=True, norm_first=True
        )
        self.predictor = nn.TransformerDecoder(decoder_layer, num_layers=depth)
        
        # Initialize weights randomly
        torch.nn.init.normal_(self.mask_token, std=.02)
        torch.nn.init.normal_(self.shared_queries, std=.02)

    def forward(self, s2_visible_features, num_hidden_patches):
        """
        s2_visible_features: Jo patches Optical mein hide nahi hue (Batch, N_Visible, Embed_Dim)
        num_hidden_patches: SAR mein kitne patches hide kiye gaye hain
        """
        batch_size = s2_visible_features.size(0)
        
        # MASK tokens ko batch size aur hidden patches ke barabar expand karein
        # Shape: (Batch, num_hidden_patches, Embed_Dim)
        mask_tokens = self.mask_token.expand(batch_size, num_hidden_patches, -1)
        
        # Shared queries ko batch ke liye expand karein
        # Shape: (Batch, num_shared_queries, Embed_Dim)
        queries = self.shared_queries.expand(batch_size, -1, -1)
        
        # Decoder input = Queries + Mask Tokens (Yehi Target sequence hai)
        decoder_input = torch.cat([queries, mask_tokens], dim=1)
        
        # 4. CROSS-ATTENTION MAGIC
        # tgt = Decoder Input (Masks + Queries), memory = Context (Optical Visible Features)
        predicted_features = self.predictor(tgt=decoder_input, memory=s2_visible_features)
        
        # Humein sirf Mask tokens ki prediction chahiye (Queries ko ignore kar denge output se)
        predicted_sar_hidden = predicted_features[:, queries.size(1):, :]
        
        return predicted_sar_hidden

# --- Test the Predictor ---
if __name__ == "__main__":
    batch_size = 32
    embed_dim = 192
    total_patches = 225
    mask_ratio = 0.5
    
    num_hidden = int(mask_ratio * total_patches) # 112 patches hidden
    num_visible = total_patches - num_hidden     # 113 patches visible
    
    # 1. Simulate the outputs from our Phase 2 ViT Encoders
    print("Simulating S2 (Optical) visible features from ViT...")
    # NOTE: Humne Global CLS token (+1) bhi visible features me add kar diya hai
    s2_visible_context = torch.randn(batch_size, num_visible + 1, embed_dim) 
    
    # 2. Initialize Predictor
    print("Initializing X-JEPA Cross-Attentional Predictor...")
    predictor = CrossAttnPredictor(embed_dim=embed_dim)
    
    # 3. Predict the SAR hidden features
    print("Predicting SAR latent features from Optical context...")
    predicted_sar_features = predictor(s2_visible_context, num_hidden)
    
    print(f"\nPredicted SAR Features Shape: {predicted_sar_features.shape}")
    print(f"Expected Shape: ({batch_size}, {num_hidden}, {embed_dim})")
    print("Phase 3 is Complete! ✅ Our model can now hallucinate SAR features from Optical data.")

Simulating S2 (Optical) visible features from ViT...
Initializing X-JEPA Cross-Attentional Predictor...
Predicting SAR latent features from Optical context...

Predicted SAR Features Shape: torch.Size([32, 112, 192])
Expected Shape: (32, 112, 192)
Phase 3 is Complete! ✅ Our model can now hallucinate SAR features from Optical data.
